# Hyperparameter sensitivity analysis (fourth revision round)
**Manuscript IJCS_260811Sc.** Adds nested cross-validation tuning and Gradient Boosting to the rev3 protocol. No rev3 number changes.

Run with *Runtime > Restart and run all*. No file upload is needed.

Outputs: `results_rev4/*.csv`, `results_rev4/self_check_rev4.json`, `run_log_rev4.txt`, and `results_rev4.zip` (download this file).

## 0. Environment
Pinned scikit-learn, versions, seeds, output folders.

In [ ]:
# =============================================================================
# SEE_IJCS_rev4_tuning - hyperparameter sensitivity analysis (fourth revision round, IJCS_260811Sc)
#
# Adds to the rev3 protocol, without changing any rev3 number:
#   (1) nested cross-validation tuning of the algorithms that have hyperparameters (DT, RF, KNN, SVR);
#   (2) Gradient Boosting (GB) as an additional reference learner (default and tuned).
# Same data, preprocessing, outer folds (10-fold, partition seeds 42-51) and metrics as SEE_IJCS_rev3.ipynb.
# Specifications: semi-log (primary) and log-log.
# Outputs: results_rev4/*.csv, results_rev4/self_check_rev4.json, run_log_rev4.txt
# =============================================================================
import os, sys, io, json, time, hashlib, platform, datetime, subprocess, warnings, importlib, urllib.request
import importlib.metadata as _md
warnings.filterwarnings("ignore")
T_START = time.time()

PINNED = {"scikit-learn": "1.8.0"}            # version of the rev3 reference run
SKIP_PIN = os.environ.get("SEE_SKIP_PIN", "0") == "1"

def _ver(pkg):
    importlib.invalidate_caches()
    try: return _md.version(pkg)
    except _md.PackageNotFoundError: return None

def _pip_install(spec):
    base = [sys.executable, "-m", "pip", "install", "-q", spec]
    for extra in ([], ["--break-system-packages"]):
        if subprocess.run(base + extra, capture_output=True, text=True).returncode == 0:
            return True
    return False

PIN_LOG = []
for _pkg, _want in PINNED.items():
    _have = _ver(_pkg)
    if _have == _want:
        PIN_LOG.append(f"{_pkg}=={_want}: already installed"); continue
    if SKIP_PIN:
        PIN_LOG.append(f"{_pkg}: pin skipped, found {_have}"); continue
    if "sklearn" in sys.modules:
        PIN_LOG.append("WARNING: sklearn was imported before pinning. Restart the runtime and run all cells again.")
    _ok = _pip_install(f"{_pkg}=={_want}")
    PIN_LOG.append(f"{_pkg}: wanted {_want}, found {_have}, now {_ver(_pkg)}" + ("" if _ok else "  (pip failed)"))
importlib.invalidate_caches()

import numpy as np, pandas as pd, scipy, sklearn
from scipy import stats
from scipy.io import arff
from sklearn.base import clone
from sklearn.model_selection import KFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR

VERSIONS = {"python": platform.python_version(), "numpy": np.__version__, "pandas": pd.__version__,
            "scipy": scipy.__version__, "scikit-learn": sklearn.__version__}
SEED0 = 42
SEEDS = list(range(42, 52))
N_SPLITS = 10          # outer folds (identical to rev3)
N_INNER = 5            # inner folds used only to select hyperparameters
RES, DATA = "results_rev4", "data"
for _d in (RES, DATA):
    os.makedirs(_d, exist_ok=True)
for _l in PIN_LOG: print("[pin]", _l)
print("[versions]", ", ".join(f"{k} {v}" for k, v in VERSIONS.items()))

## II-A. Datasets
Identical to `SEE_IJCS_rev3.ipynb` (download, md5 check, preprocessing).

In [ ]:
# =============================================================================
# II-A. DATASETS - identical to SEE_IJCS_rev3.ipynb
# =============================================================================
ORDINAL = {"vl": 1, "l": 2, "n": 3, "h": 4, "vh": 5, "xh": 6}
DRIVERS = ["rely", "data", "cplx", "time", "stor", "virt", "turn", "acap", "aexp", "pcap", "vexp", "lexp", "modp", "tool", "sced"]
_OURMINE = "https://raw.githubusercontent.com/timm/ourmine/master/our/arffs/effest/"
DATASETS = {
    "COCOMO81": dict(file="coc81_1_1.arff", url=_OURMINE + "coc81_1_1.arff", md5="2898f9dca7709ba5dc6384500e97a5f6",
                     target="actual", unit="person-months", drop=["project_id", "dev_mode"], onehot=[], ordinal=[],
                     size=["loc"], columns=DRIVERS + ["loc"]),
    "NASA93": dict(file="nasa93.arff", url=_OURMINE + "nasa93.arff", md5="6bfcb7491cb8c592a7e719a50e086167",
                   target="act_effort", unit="person-months",
                   drop=["recordnumber", "projectname", "cat2", "forg", "center", "year"], onehot=["mode"], ordinal=DRIVERS,
                   size=["equivphyskloc"], columns=DRIVERS + ["equivphyskloc", "mode_organic", "mode_semidetached"]),
    "Desharnais": dict(file="desharnais.arff.txt",
                       url="https://raw.githubusercontent.com/deepak21188/Software-Development-Effort-Estimation-using-ML/master/desharnais.arff.txt",
                       md5="59c5b1a1978986239793ea4c78c504fd", target="Effort", unit="person-hours",
                       drop=["Project"], onehot=["Language"], ordinal=[], size=["PointsAdjust", "PointsNonAjust"],
                       columns=["TeamExp", "ManagerExp", "YearEnd", "Length", "Transactions", "Entities",
                                "PointsAdjust", "Envergure", "PointsNonAjust", "Language_2", "Language_3"]),
}
DS_LIST = list(DATASETS)

def _resolve(name, cols):
    return {c.lower(): c for c in cols}.get(name.lower())

def fetch(cfg):
    path = os.path.join(DATA, cfg["file"])
    if not os.path.exists(path):
        req = urllib.request.Request(cfg["url"], headers={"User-Agent": "Mozilla/5.0"})
        with open(path, "wb") as fh:
            fh.write(urllib.request.urlopen(req, timeout=60).read())
    raw = open(path, "rb").read()
    return raw, hashlib.md5(raw).hexdigest()

def load_dataset(name):
    cfg = DATASETS[name]
    raw, digest = fetch(cfg)
    assert digest == cfg["md5"], f"{name}: md5 {digest} differs from the reference {cfg['md5']}"
    rec, _ = arff.loadarff(io.StringIO(raw.decode("utf-8", "replace")))
    df = pd.DataFrame(rec)
    df.columns = [c.strip() for c in df.columns]
    for c in df.columns:
        if not pd.api.types.is_numeric_dtype(df[c]):
            df[c] = df[c].map(lambda v: v.decode() if isinstance(v, (bytes, bytearray)) else v).replace("?", np.nan)
    for col in cfg["ordinal"]:
        cc = _resolve(col, df.columns)
        mapped = df[cc].astype(str).str.strip().str.lower().map(ORDINAL)
        assert not mapped.isna().any(), f"{name}: unmapped ordinal level in {cc}"
        df[cc] = mapped
    df = df.dropna().reset_index(drop=True)
    target = _resolve(cfg["target"], df.columns)
    y = df[target].astype(float).values
    dropped = [_resolve(c, df.columns) for c in cfg["drop"]]
    X = df.drop(columns=[target] + dropped)
    if cfg["onehot"]:
        X = pd.get_dummies(X, columns=[_resolve(c, X.columns) for c in cfg["onehot"]], drop_first=True)
    X = X.apply(pd.to_numeric, errors="coerce")
    assert not X.isna().any().any()
    assert list(X.columns) == cfg["columns"], f"{name}: unexpected column order {list(X.columns)}"
    return X.astype(float), y

from decimal import Decimal, getcontext
getcontext().prec = 50
def log1p_exact(values):
    """Correctly rounded ln(1 + v), platform independent (see SEE_IJCS_rev3.ipynb)."""
    flat = [float((Decimal(1) + Decimal(float(v))).ln()) for v in np.ravel(values)]
    return np.array(flat, dtype=float).reshape(np.shape(values))

DATA_X, DATA_Y, Y_LOG, _desc = {}, {}, {}, []
for ds in DS_LIST:
    X, y = load_dataset(ds)
    DATA_X[ds], DATA_Y[ds], Y_LOG[ds] = X, y, log1p_exact(y)
    row = dict(dataset=ds, n=len(y), features=X.shape[1], effort_min=float(y.min()), effort_median=float(np.median(y)),
               effort_mean=float(y.mean()), effort_sd=float(y.std(ddof=1)), effort_max=float(y.max()),
               skewness=float(stats.skew(y)), skewness_log1p=float(stats.skew(Y_LOG[ds])))
    s = DATASETS[ds]["size"][0]
    row.update(size_feature=s, size_min=float(X[s].min()), size_median=float(X[s].median()), size_max=float(X[s].max()))
    _desc.append(row)
DESCRIPTIVES = pd.DataFrame(_desc)
print(DESCRIPTIVES.round(2).to_string(index=False))

## II-C. Metrics
Identical to `SEE_IJCS_rev3.ipynb`.

In [ ]:
# =============================================================================
# II-C. METRICS - identical to SEE_IJCS_rev3.ipynb
# =============================================================================
METRICS = ["MAE", "RMSE", "MMRE", "MdMRE", "PRED25", "SA"]
PRED_TOL = 1e-9
def fold_metrics(y_true, y_pred, y_train):
    ae = np.abs(y_true - y_pred)
    mre = ae / np.clip(np.abs(y_true), 1e-9, None)
    mae_p0 = np.mean([np.mean(np.abs(yi - y_train)) for yi in y_true])
    return dict(MAE=float(ae.mean()), RMSE=float(np.sqrt(np.mean((y_true - y_pred) ** 2))),
                MMRE=float(mre.mean()), MdMRE=float(np.median(mre)),
                PRED25=float(100.0 * np.mean(mre <= 0.25 + PRED_TOL)), SA=float(100.0 * (1.0 - ae.mean() / mae_p0)))

## II-D. Protocol: default settings versus nested-CV tuning
This cell takes about 30 minutes in Colab.

In [ ]:
# =============================================================================
# II-D. PROTOCOL - default settings (rev3) versus nested-CV tuning
# =============================================================================
ALGOS = ["LR", "DT", "RF", "KNN", "SVR", "GB"]
SPECS = {"semilog": dict(log_size=False), "loglog": dict(log_size=True)}     # the target is ln(1+y) in both

def make_models():       # LR, DT, RF, KNN, SVR exactly as in rev3; GB is the added reference learner
    return {"LR": LinearRegression(), "DT": DecisionTreeRegressor(random_state=SEED0),
            "RF": RandomForestRegressor(n_estimators=100, random_state=SEED0),
            "KNN": KNeighborsRegressor(n_neighbors=5), "SVR": SVR(kernel="rbf", C=1.0, epsilon=0.1),
            "GB": GradientBoostingRegressor(random_state=SEED0)}

# Search grids (every grid contains the default setting). LR has no hyperparameter and is not tuned.
GRIDS = {
    "DT":  {"model__max_depth": [None, 3, 5], "model__min_samples_leaf": [1, 3, 5]},
    "RF":  {"model__max_features": [1.0, 0.5], "model__min_samples_leaf": [1, 3]},
    "KNN": {"model__n_neighbors": [1, 3, 5, 7, 9], "model__weights": ["uniform", "distance"]},
    "SVR": {"model__C": [0.1, 1.0, 10.0, 100.0], "model__epsilon": [0.01, 0.1, 0.5], "model__gamma": ["scale", 0.01, 0.1]},
    "GB":  {"model__learning_rate": [0.1, 0.05], "model__max_depth": [3, 2]},
}

def make_estimator(model):
    return Pipeline([("scale", StandardScaler()), ("model", model)])

def design_matrix(Xdf, size_cols, log_size):
    X = Xdf.values.astype(float).copy()
    if log_size:
        idx = [list(Xdf.columns).index(c) for c in size_cols]
        X[:, idx] = log1p_exact(X[:, idx])
    return X

def run_cv(X, y, y_log, seed):
    """Outer folds are those of rev3. In the tuned arm, hyperparameters are selected inside each training fold
    by 5-fold cross-validation (lowest MAE on the ln(1+effort) scale); the test fold is never seen during selection."""
    folds = list(KFold(n_splits=N_SPLITS, shuffle=True, random_state=seed).split(X))
    out, chosen = [], []
    for a, model in make_models().items():
        for arm in ("default", "tuned"):
            if arm == "tuned" and a not in GRIDS:
                continue
            per_fold = {m: [] for m in METRICS}
            for k, (tr, te) in enumerate(folds):
                est = make_estimator(clone(model))
                if arm == "tuned":
                    inner = KFold(n_splits=N_INNER, shuffle=True, random_state=seed)
                    est = GridSearchCV(est, GRIDS[a], scoring="neg_mean_absolute_error", cv=inner, refit=True)
                est.fit(X[tr], y_log[tr])
                pred = np.expm1(est.predict(X[te]))
                for m, v in fold_metrics(y[te], pred, y[tr]).items():
                    per_fold[m].append(v)
                if arm == "tuned":
                    chosen.append(dict(algorithm=a, seed=seed, fold=k + 1,
                                       **{p.replace("model__", ""): str(v) for p, v in est.best_params_.items()}))
            out.append(dict(algorithm=a, arm=arm, seed=seed, **{m: float(np.mean(per_fold[m])) for m in METRICS}))
    return out, chosen

rows, chosen_rows = [], []
_t = time.time()
for ds in DS_LIST:
    for spec, opt in SPECS.items():
        X = design_matrix(DATA_X[ds], DATASETS[ds]["size"], opt["log_size"])
        for seed in SEEDS:
            out, chosen = run_cv(X, DATA_Y[ds], Y_LOG[ds], seed)
            rows += [dict(dataset=ds, spec=spec, **r) for r in out]
            chosen_rows += [dict(dataset=ds, spec=spec, **r) for r in chosen]
        print(f"{ds} / {spec}: {len(SEEDS)} partitions done ({time.time() - _t:.0f} s elapsed)", flush=True)
PER_SEED = pd.DataFrame(rows)
CHOSEN = pd.DataFrame(chosen_rows)

## III. Result tables

In [ ]:
# =============================================================================
# III. RESULT TABLES
# =============================================================================
# Mean and sample SD (ddof = 1) over the ten partitions
SUMMARY = (PER_SEED.groupby(["dataset", "spec", "algorithm", "arm"], sort=False)[METRICS]
           .agg(["mean", lambda v: v.std(ddof=1), "min", "max"]))
SUMMARY.columns = [f"{m}_{s}" for m, s in zip([c[0] for c in SUMMARY.columns], ["mean10", "sd10", "min10", "max10"] * len(METRICS))]
SUMMARY = SUMMARY.reset_index()

# Paired comparison tuned vs default over the ten partitions (same outer folds)
cmp_rows = []
for (ds, spec, a), g in PER_SEED.groupby(["dataset", "spec", "algorithm"], sort=False):
    if a not in GRIDS: continue
    d = g[g.arm == "default"].sort_values("seed")["MAE"].values
    t = g[g.arm == "tuned"].sort_values("seed")["MAE"].values
    try: p = float(stats.wilcoxon(t, d).pvalue)
    except ValueError: p = 1.0
    cmp_rows.append(dict(dataset=ds, spec=spec, algorithm=a, MAE_default_mean10=float(d.mean()), MAE_tuned_mean10=float(t.mean()),
                         change_percent=float(100 * (t.mean() - d.mean()) / d.mean()), partitions_tuned_lower=int((t < d).sum()),
                         wilcoxon_p=p))
TUNED_VS_DEFAULT = pd.DataFrame(cmp_rows)

# Ranking by mean MAE over the ten partitions: does the conclusion on the best learner change?
rank_rows = []
for (ds, spec), g in SUMMARY.groupby(["dataset", "spec"], sort=False):
    for label, sel in (("rev3 (five learners, default)", g[(g.arm == "default") & (g.algorithm != "GB")]),
                       ("default, with GB", g[g.arm == "default"]),
                       ("tuned where applicable, with GB", pd.concat([g[g.arm == "tuned"], g[(g.algorithm == "LR")]]))):
        s = sel.sort_values("MAE_mean10")
        rank_rows.append(dict(dataset=ds, spec=spec, setting=label, best=s.iloc[0]["algorithm"], best_MAE=float(s.iloc[0]["MAE_mean10"]),
                              second=s.iloc[1]["algorithm"], second_MAE=float(s.iloc[1]["MAE_mean10"]),
                              order=" < ".join(s["algorithm"])))
RANKING = pd.DataFrame(rank_rows)

# Number of partitions (of ten) in which each learner has the lowest MAE, tuned setting
win_rows = []
for (ds, spec), g in PER_SEED.groupby(["dataset", "spec"], sort=False):
    sel = pd.concat([g[g.arm == "tuned"], g[g.algorithm == "LR"]])
    piv = sel.pivot(index="seed", columns="algorithm", values="MAE")
    w = piv.idxmin(axis=1).value_counts()
    win_rows.append(dict(dataset=ds, spec=spec, **{a: int(w.get(a, 0)) for a in ALGOS}))
WINS_TUNED = pd.DataFrame(win_rows)

# Most frequently selected hyperparameters (over 10 partitions x 10 outer folds)
sel_rows = []
for (ds, spec, a), g in CHOSEN.groupby(["dataset", "spec", "algorithm"], sort=False):
    cols = [c.replace("model__", "") for c in GRIDS[a]]
    combo = g[cols].astype(str).agg(", ".join, axis=1)
    vc = combo.value_counts()
    top = sorted(vc[vc == vc.iloc[0]].index)[0]
    sel_rows.append(dict(dataset=ds, spec=spec, algorithm=a, parameters=", ".join(cols), most_selected=top,
                         share_percent=float(100 * vc[top] / len(g)), distinct_settings_selected=int(len(vc))))
SELECTED = pd.DataFrame(sel_rows)

pd.set_option("display.width", 250); pd.set_option("display.float_format", lambda v: f"{v:,.2f}")
print("\nMAE, mean over ten partitions:")
print(SUMMARY.pivot_table(index=["dataset", "spec", "algorithm"], columns="arm", values="MAE_mean10", sort=False).to_string())
print("\nBest learner by mean MAE:"); print(RANKING[["dataset", "spec", "setting", "best", "best_MAE", "second", "second_MAE"]].to_string(index=False))

## Export and self-check
The last lines print the digests and compare them with the reference run.

In [ ]:
# =============================================================================
# EXPORT AND SELF-CHECK
# =============================================================================
for name, df in (("per_seed_rev4", PER_SEED), ("summary_rev4", SUMMARY), ("tuned_vs_default", TUNED_VS_DEFAULT), ("ranking", RANKING),
                 ("wins_tuned", WINS_TUNED), ("selected_hyperparameters", SELECTED), ("chosen_per_fold", CHOSEN), ("descriptives", DESCRIPTIVES)):
    df.to_csv(os.path.join(RES, name + ".csv"), index=False)

def _sig(v):
    v = float(v)
    return "nan" if np.isnan(v) else f"{v:.6g}"
def _digest(frames):
    vals = []
    for df in frames:
        vals += [_sig(v) for v in df.select_dtypes(include=[np.number]).values.ravel()]
    return hashlib.sha256(json.dumps(vals).encode()).hexdigest()[:16]

_def5 = PER_SEED[(PER_SEED.arm == "default") & (PER_SEED.algorithm != "GB")]
DIGEST_DEFAULT5 = _digest([_def5])                                        # the part that must equal rev3
DIGEST_GB_DEFAULT = _digest([PER_SEED[(PER_SEED.arm == "default") & (PER_SEED.algorithm == "GB")]])
DIGEST_TUNED = {a: _digest([PER_SEED[(PER_SEED.arm == "tuned") & (PER_SEED.algorithm == a)]]) for a in GRIDS}
DIGEST_ALL = _digest([PER_SEED])

# Headline numbers of rev3 that this script must reproduce (LR, log-log, mean MAE over ten partitions)
REV3 = {"COCOMO81": 282.03676231659495, "NASA93": 254.69377397640764, "Desharnais": 1809.1202746157985}
rev3_ok = all(abs(SUMMARY[(SUMMARY.dataset == d) & (SUMMARY.spec == "loglog") & (SUMMARY.algorithm == "LR") & (SUMMARY.arm == "default")]["MAE_mean10"].iloc[0] - v) <= 1e-6 * v
              for d, v in REV3.items())

REFERENCE = json.loads(r'''{"digest_default5": "8c0755c03e052934", "digest_gb_default": "c52972156e5824e8", "digest_tuned": {"DT": "50ff210ae90a403a", "RF": "c5013a85797a5167", "KNN": "56cb24b97220ec46", "SVR": "2d3d1831ebaa441f", "GB": "55c8054118deaefb"}, "digest_all": "6f648753968645d9", "mae_mean10": [{"dataset": "COCOMO81", "spec": "semilog", "algorithm": "LR", "arm": "default", "ref": 195566.6927181025}, {"dataset": "COCOMO81", "spec": "semilog", "algorithm": "DT", "arm": "default", "ref": 564.6267380952384}, {"dataset": "COCOMO81", "spec": "semilog", "algorithm": "DT", "arm": "tuned", "ref": 548.1738116543703}, {"dataset": "COCOMO81", "spec": "semilog", "algorithm": "RF", "arm": "default", "ref": 442.83133973070187}, {"dataset": "COCOMO81", "spec": "semilog", "algorithm": "RF", "arm": "tuned", "ref": 473.78766651601245}, {"dataset": "COCOMO81", "spec": "semilog", "algorithm": "KNN", "arm": "default", "ref": 579.4657144190462}, {"dataset": "COCOMO81", "spec": "semilog", "algorithm": "KNN", "arm": "tuned", "ref": 581.3341542288738}, {"dataset": "COCOMO81", "spec": "semilog", "algorithm": "SVR", "arm": "default", "ref": 592.0890881365134}, {"dataset": "COCOMO81", "spec": "semilog", "algorithm": "SVR", "arm": "tuned", "ref": 422.3748200618188}, {"dataset": "COCOMO81", "spec": "semilog", "algorithm": "GB", "arm": "default", "ref": 384.56951295223445}, {"dataset": "COCOMO81", "spec": "semilog", "algorithm": "GB", "arm": "tuned", "ref": 350.76982047695395}, {"dataset": "COCOMO81", "spec": "loglog", "algorithm": "LR", "arm": "default", "ref": 282.03676231659495}, {"dataset": "COCOMO81", "spec": "loglog", "algorithm": "DT", "arm": "default", "ref": 564.9793571428573}, {"dataset": "COCOMO81", "spec": "loglog", "algorithm": "DT", "arm": "tuned", "ref": 556.4879334116738}, {"dataset": "COCOMO81", "spec": "loglog", "algorithm": "RF", "arm": "default", "ref": 442.490982677394}, {"dataset": "COCOMO81", "spec": "loglog", "algorithm": "RF", "arm": "tuned", "ref": 475.62604209796353}, {"dataset": "COCOMO81", "spec": "loglog", "algorithm": "KNN", "arm": "default", "ref": 598.0582310199175}, {"dataset": "COCOMO81", "spec": "loglog", "algorithm": "KNN", "arm": "tuned", "ref": 576.6036801188396}, {"dataset": "COCOMO81", "spec": "loglog", "algorithm": "SVR", "arm": "default", "ref": 543.8471392666238}, {"dataset": "COCOMO81", "spec": "loglog", "algorithm": "SVR", "arm": "tuned", "ref": 292.2216515878253}, {"dataset": "COCOMO81", "spec": "loglog", "algorithm": "GB", "arm": "default", "ref": 385.98172289484455}, {"dataset": "COCOMO81", "spec": "loglog", "algorithm": "GB", "arm": "tuned", "ref": 351.20447970910504}, {"dataset": "NASA93", "spec": "semilog", "algorithm": "LR", "arm": "default", "ref": 1897.9539213762498}, {"dataset": "NASA93", "spec": "semilog", "algorithm": "DT", "arm": "default", "ref": 412.20206194529965}, {"dataset": "NASA93", "spec": "semilog", "algorithm": "DT", "arm": "tuned", "ref": 393.24636166226014}, {"dataset": "NASA93", "spec": "semilog", "algorithm": "RF", "arm": "default", "ref": 298.59888058667866}, {"dataset": "NASA93", "spec": "semilog", "algorithm": "RF", "arm": "tuned", "ref": 317.61155942171547}, {"dataset": "NASA93", "spec": "semilog", "algorithm": "KNN", "arm": "default", "ref": 429.51520544176253}, {"dataset": "NASA93", "spec": "semilog", "algorithm": "KNN", "arm": "tuned", "ref": 429.90901244490414}, {"dataset": "NASA93", "spec": "semilog", "algorithm": "SVR", "arm": "default", "ref": 388.41063247010703}, {"dataset": "NASA93", "spec": "semilog", "algorithm": "SVR", "arm": "tuned", "ref": 496.1405129359966}, {"dataset": "NASA93", "spec": "semilog", "algorithm": "GB", "arm": "default", "ref": 289.87839820694956}, {"dataset": "NASA93", "spec": "semilog", "algorithm": "GB", "arm": "tuned", "ref": 294.5218619122121}, {"dataset": "NASA93", "spec": "loglog", "algorithm": "LR", "arm": "default", "ref": 254.69377397640764}, {"dataset": "NASA93", "spec": "loglog", "algorithm": "DT", "arm": "default", "ref": 413.9613063897441}, {"dataset": "NASA93", "spec": "loglog", "algorithm": "DT", "arm": "tuned", "ref": 393.2230421847886}, {"dataset": "NASA93", "spec": "loglog", "algorithm": "RF", "arm": "default", "ref": 297.69073365469416}, {"dataset": "NASA93", "spec": "loglog", "algorithm": "RF", "arm": "tuned", "ref": 314.68805176142234}, {"dataset": "NASA93", "spec": "loglog", "algorithm": "KNN", "arm": "default", "ref": 441.5019168944332}, {"dataset": "NASA93", "spec": "loglog", "algorithm": "KNN", "arm": "tuned", "ref": 393.59935710259185}, {"dataset": "NASA93", "spec": "loglog", "algorithm": "SVR", "arm": "default", "ref": 344.8565624150391}, {"dataset": "NASA93", "spec": "loglog", "algorithm": "SVR", "arm": "tuned", "ref": 221.95720551175341}, {"dataset": "NASA93", "spec": "loglog", "algorithm": "GB", "arm": "default", "ref": 287.32327572089457}, {"dataset": "NASA93", "spec": "loglog", "algorithm": "GB", "arm": "tuned", "ref": 294.89224241021213}, {"dataset": "Desharnais", "spec": "semilog", "algorithm": "LR", "arm": "default", "ref": 2001.875349802315}, {"dataset": "Desharnais", "spec": "semilog", "algorithm": "DT", "arm": "default", "ref": 2506.6310714285714}, {"dataset": "Desharnais", "spec": "semilog", "algorithm": "DT", "arm": "tuned", "ref": 2275.7520791346337}, {"dataset": "Desharnais", "spec": "semilog", "algorithm": "RF", "arm": "default", "ref": 1982.8086078580468}, {"dataset": "Desharnais", "spec": "semilog", "algorithm": "RF", "arm": "tuned", "ref": 1987.6905233964485}, {"dataset": "Desharnais", "spec": "semilog", "algorithm": "KNN", "arm": "default", "ref": 1981.0103655070805}, {"dataset": "Desharnais", "spec": "semilog", "algorithm": "KNN", "arm": "tuned", "ref": 1992.463426417776}, {"dataset": "Desharnais", "spec": "semilog", "algorithm": "SVR", "arm": "default", "ref": 1958.4339568013802}, {"dataset": "Desharnais", "spec": "semilog", "algorithm": "SVR", "arm": "tuned", "ref": 2022.2418239716958}, {"dataset": "Desharnais", "spec": "semilog", "algorithm": "GB", "arm": "default", "ref": 2112.5896427128455}, {"dataset": "Desharnais", "spec": "semilog", "algorithm": "GB", "arm": "tuned", "ref": 2022.3491043790884}, {"dataset": "Desharnais", "spec": "loglog", "algorithm": "LR", "arm": "default", "ref": 1809.1202746157983}, {"dataset": "Desharnais", "spec": "loglog", "algorithm": "DT", "arm": "default", "ref": 2503.8260714285716}, {"dataset": "Desharnais", "spec": "loglog", "algorithm": "DT", "arm": "tuned", "ref": 2275.291780245081}, {"dataset": "Desharnais", "spec": "loglog", "algorithm": "RF", "arm": "default", "ref": 1982.285524927304}, {"dataset": "Desharnais", "spec": "loglog", "algorithm": "RF", "arm": "tuned", "ref": 1990.710631639909}, {"dataset": "Desharnais", "spec": "loglog", "algorithm": "KNN", "arm": "default", "ref": 1935.6612795045166}, {"dataset": "Desharnais", "spec": "loglog", "algorithm": "KNN", "arm": "tuned", "ref": 2023.4679358298733}, {"dataset": "Desharnais", "spec": "loglog", "algorithm": "SVR", "arm": "default", "ref": 1916.4715258293902}, {"dataset": "Desharnais", "spec": "loglog", "algorithm": "SVR", "arm": "tuned", "ref": 1880.3247532323949}, {"dataset": "Desharnais", "spec": "loglog", "algorithm": "GB", "arm": "default", "ref": 2111.971379124781}, {"dataset": "Desharnais", "spec": "loglog", "algorithm": "GB", "arm": "tuned", "ref": 2029.627227800016}]}''')
lines = [f"digest_default5 = {DIGEST_DEFAULT5}", f"digest_gb_default = {DIGEST_GB_DEFAULT}",
         *[f"digest_tuned[{a}] = {d}" for a, d in DIGEST_TUNED.items()], f"digest_all = {DIGEST_ALL}",
         f"SELF-CHECK rev3 headline numbers (LR, log-log): {'PASS' if rev3_ok else 'FAIL'}"]
if REFERENCE:
    lines.append(f"SELF-CHECK default arm, five rev3 learners: {'PASS' if DIGEST_DEFAULT5 == REFERENCE['digest_default5'] else 'DIFFERENT'}")
    lines.append(f"SELF-CHECK GB default: {'PASS' if DIGEST_GB_DEFAULT == REFERENCE['digest_gb_default'] else 'DIFFERENT'}")
    for a in GRIDS:
        lines.append(f"SELF-CHECK tuned {a}: {'PASS' if DIGEST_TUNED[a] == REFERENCE['digest_tuned'][a] else 'DIFFERENT'}")
    lines.append(f"SELF-CHECK all numbers: {'PASS' if DIGEST_ALL == REFERENCE['digest_all'] else 'DIFFERENT'}")
    # Largest relative difference from the reference run in the reported table (mean MAE over ten partitions)
    ref = pd.DataFrame(REFERENCE["mae_mean10"])
    mrg = SUMMARY.merge(ref, on=["dataset", "spec", "algorithm", "arm"])
    rel = (mrg["MAE_mean10"] - mrg["ref"]).abs() / mrg["ref"]
    lines.append(f"largest relative difference in mean MAE from the reference run: {rel.max():.3e} ({len(mrg)} cells)")
json.dump(dict(digest_default5=DIGEST_DEFAULT5, digest_gb_default=DIGEST_GB_DEFAULT, digest_tuned=DIGEST_TUNED, digest_all=DIGEST_ALL,
               mae_mean10=[dict(dataset=r.dataset, spec=r.spec, algorithm=r.algorithm, arm=r.arm, ref=r.MAE_mean10) for r in SUMMARY.itertuples()],
               versions=VERSIONS), open(os.path.join(RES, "self_check_rev4.json"), "w"), indent=1)
runtime = time.time() - T_START
log = ["SEE_IJCS_rev4_tuning run log", f"run finished (UTC): {datetime.datetime.utcnow().isoformat(timespec='seconds')}",
       f"runtime: {runtime:.0f} s", f"platform: {platform.platform()}", *[f"{k}: {v}" for k, v in VERSIONS.items()],
       *[f"[pin] {l}" for l in PIN_LOG], f"outer folds: {N_SPLITS}; inner folds: {N_INNER}; partitions: {SEEDS}", *lines]
open("run_log_rev4.txt", "w").write("\n".join(log) + "\n")
print("\n" + "\n".join(lines)); print(f"runtime: {runtime:.0f} s")

In [ ]:
import shutil
shutil.copy('run_log_rev4.txt', 'results_rev4/run_log_rev4.txt')
shutil.make_archive('results_rev4', 'zip', 'results_rev4')
print(open('run_log_rev4.txt').read())
try:
    from google.colab import files; files.download('results_rev4.zip')
except Exception:
    print('results_rev4.zip written')
